# Spatial Branch — Final Training (From Scratch)

## Plan
- Train from scratch — no patching old model
- Real class: FF++ Real + LFW (covers all real domains)
- Fake class: FF++ 6 manipulation types only
- Phone images: held-out test only (not face crops — wrong format for training)
- CelebDF: held-out cross-dataset test only
- Improved augmentation including random erasing
- No pos_weight, no sampler — diversity of real sources is the fix
- Pareto threshold after training (defensible, standard practice)

In [ ]:
import os, io, gc, random
from pathlib import Path
from dataclasses import dataclass
from collections import Counter

import numpy as np
from PIL import Image, ImageFilter, ImageEnhance
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_curve, auc
)
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# ══════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════
SEED        = 42
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

# Training from scratch — full LR, full epochs
START_EPOCH = 0
EPOCHS      = 15
BATCH_SIZE  = 32
LR          = 1e-4
LR_MIN      = 1e-6
WEIGHT_DECAY= 1e-4
GRAD_CLIP   = 1.0
# No pos_weight — real diversity handles imbalance

TARGET_SIZE   = 224
RESIZE_SIZE   = 256
IMG_EXTS      = {'.jpg','.jpeg','.png','.webp','.bmp'}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

FF_CAP_PER_TYPE = None   # use all FF++ images
LFW_CAP         = 13000  # full LFW
CELEB_FAKE_CAP  = 15000
CELEB_REAL_CAP  = 8000
FAST_VAL_N      = 4000

CKPT_LOAD = None   # training from scratch
CKPT_DIR  = '/kaggle/working'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print('Device:', DEVICE)
print('Training from scratch — START_EPOCH=0')
print('Config OK')

In [ ]:
# ══════════════════════════════════════════
#  PATHS
#
#  Real sources for training:
#    FF++ Real — controlled lab video frames
#    LFW       — diverse uncontrolled real faces
#                (linear probe proved: 98.86% recall on phone images)
#
#  Phone images (Nathia Gali): NOT used in training
#    Reason: full-body outdoor shots, not face crops
#    Use case: held-out test after fusion
#
#  CelebDF: held-out cross-dataset test only
# ══════════════════════════════════════════
FF_SPLIT_ROOT = Path('/kaggle/input/datasets/gradientvoyager/faceforensics-c23-extracted-faces-100k/dataset_processed_split')
LFW_ROOT      = Path('/kaggle/input/datasets/jessicali9530/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled')
CELEB_ROOT    = Path('/kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2')

FF_FAKE_TYPES = [
    'Deepfakes', 'Face2Face', 'FaceShifter',
    'FaceSwap', 'NeuralTextures', 'DeepFakeDetection'
]
FF_REAL_NAME = 'Real'

for name, p in [
    ('FF++ split root', FF_SPLIT_ROOT),
    ('LFW root',        LFW_ROOT),
    ('CelebDF root',    CELEB_ROOT),
]:
    print(f'  {name:20s}: {"OK" if p.exists() else "MISSING"}')

print('\nFF++ image counts:')
for split in ['train','val','test']:
    sd = FF_SPLIT_ROOT / split
    if not sd.exists(): continue
    print(f'  {split}/')
    for folder in sorted(sd.iterdir()):
        if folder.is_dir():
            n = sum(1 for f in folder.rglob('*')
                    if f.suffix.lower() in IMG_EXTS)
            print(f'    {folder.name}: {n:,}')

In [ ]:
@dataclass(frozen=True)
class SampleRef:
    path:   str
    label:  int    # 0=real  1=fake
    source: str

def list_images(p: Path):
    if not p.exists(): return []
    return [x for x in p.rglob('*')
            if x.is_file() and x.suffix.lower() in IMG_EXTS]

def cap_shuffle(paths, cap):
    random.shuffle(paths)
    return paths[:cap] if cap and len(paths) > cap else paths

print('SampleRef defined.')

In [ ]:
# ══════════════════════════════════════════
#  BUILD FF++ + LFW SPLITS
#
#  FF++ split used EXACTLY as provided.
#  Never re-split FF++ randomly — causes leakage
#  (frames from same video would appear in train AND test).
#
#  LFW split by identity (person-disjoint splits):
#  All images of one person stay in same split.
#  This is the correct way — if same person appears in
#  both train and test, accuracy is inflated.
# ══════════════════════════════════════════
train_refs, val_refs, test_refs = [], [], []

for split_name, ref_list in [('train', train_refs),
                              ('val',   val_refs),
                              ('test',  test_refs)]:
    sd = FF_SPLIT_ROOT / split_name
    if not sd.exists():
        print(f'WARNING: {sd} not found'); continue

    # FF++ Real
    rp = cap_shuffle(list_images(sd / FF_REAL_NAME), FF_CAP_PER_TYPE)
    ref_list += [SampleRef(str(p), 0, 'ff_Real') for p in rp]

    # FF++ Fake — all 6 manipulation types
    for ft in FF_FAKE_TYPES:
        fd = sd / ft
        if not fd.exists():
            print(f'  SKIP {ft} — not found'); continue
        fp = cap_shuffle(list_images(fd), FF_CAP_PER_TYPE)
        ref_list += [SampleRef(str(p), 1, f'ff_{ft}') for p in fp]

# LFW: identity-disjoint split 75/10/15
all_ids = sorted([d for d in LFW_ROOT.iterdir() if d.is_dir()])
random.shuffle(all_ids)
n      = len(all_ids)
nt, nv = int(0.75*n), int(0.10*n)

def lfw_from_ids(id_dirs, cap):
    paths = []
    for d in id_dirs:
        paths += list_images(d)
    return cap_shuffle(paths, cap)

lfw_tr = lfw_from_ids(all_ids[:nt],       int(LFW_CAP*0.75))
lfw_va = lfw_from_ids(all_ids[nt:nt+nv],  int(LFW_CAP*0.10))
lfw_te = lfw_from_ids(all_ids[nt+nv:],    LFW_CAP - int(LFW_CAP*0.75) - int(LFW_CAP*0.10))

train_refs += [SampleRef(str(p), 0, 'lfw_real') for p in lfw_tr]
val_refs   += [SampleRef(str(p), 0, 'lfw_real') for p in lfw_va]
test_refs  += [SampleRef(str(p), 0, 'lfw_real') for p in lfw_te]

random.shuffle(train_refs)
random.shuffle(val_refs)
random.shuffle(test_refs)

print('Splits:')
for name, refs in [('Train',train_refs),('Val',val_refs),('Test',test_refs)]:
    lc = Counter(r.label  for r in refs)
    sc = Counter(r.source for r in refs)
    print(f'\n  {name}: {len(refs):,}  real={lc[0]:,}  fake={lc[1]:,}  ratio={lc[1]/max(lc[0],1):.1f}:1')
    for s,c in sorted(sc.items()): print(f'    {s}: {c:,}')

In [ ]:
# CelebDF — held-out cross-dataset test, never trained on
celeb_fake_paths, celeb_real_paths = [], []
for sn in ['Train','Val','Test']:
    sd = CELEB_ROOT / sn
    if not sd.exists(): continue
    for fn in ['fake','Fake']:
        fd = sd/fn
        if fd.exists(): celeb_fake_paths += list_images(fd); break
    for rn in ['real','Real']:
        rd = sd/rn
        if rd.exists(): celeb_real_paths += list_images(rd); break

print(f'CelebDF raw: real={len(celeb_real_paths):,}  fake={len(celeb_fake_paths):,}')
celeb_fake_paths = cap_shuffle(celeb_fake_paths, CELEB_FAKE_CAP)
celeb_real_paths = cap_shuffle(celeb_real_paths, CELEB_REAL_CAP)
celeb_refs  = [SampleRef(str(p),1,'celebdf_fake') for p in celeb_fake_paths]
celeb_refs += [SampleRef(str(p),0,'celebdf_real') for p in celeb_real_paths]
random.shuffle(celeb_refs)
cc = Counter(r.label for r in celeb_refs)
print(f'CelebDF test set: {len(celeb_refs):,}  real={cc[0]:,}  fake={cc[1]:,}')

In [ ]:
# ══════════════════════════════════════════
#  AUGMENTATION — THE ACTUAL BIAS FIX
#
#  Your friend's experience: improved preprocessing
#  removed bias better than any architecture change.
#  Here is why:
#
#  Without strong augmentation, the model memorizes
#  dataset-specific patterns:
#  'FF++ real = specific compression + lighting'
#  'FF++ fake = specific manipulation texture'
#  When it sees a phone image, it has no FF++ real
#  statistics to match, so it defaults to fake.
#
#  With strong augmentation, those statistics are
#  destroyed during training. The model is forced
#  to learn genuine discriminative features:
#  - Blending boundaries (face swap artifacts)
#  - Texture inconsistencies (reenactment artifacts)
#  - Identity-level anomalies
#  These generalize across domains.
#
#  New vs v1: wider blur/JPEG, brightness/contrast/
#  saturation jitter, grayscale, sharpening, random erasing
#
#  ORDER: Geometric -> Photometric -> Degradation -> Structural
#  Applied AFTER resize+crop, BEFORE tensor conversion
# ══════════════════════════════════════════
def random_erase(img: Image.Image, p=0.3) -> Image.Image:
    """
    Randomly erases a rectangular patch (2-15% of image area).
    Forces model to learn local artifacts, not global face structure.
    Key for removing FF++ memorization bias.
    """
    if random.random() >= p:
        return img
    w, h  = img.size
    area  = w * h
    # patch covers 2-15% of image
    patch_area = random.uniform(0.02, 0.15) * area
    ratio      = random.uniform(0.3, 3.0)
    ph = int((patch_area / ratio) ** 0.5)
    pw = int(ph * ratio)
    ph = min(ph, h)
    pw = min(pw, w)
    x0 = random.randint(0, max(1, w - pw))
    y0 = random.randint(0, max(1, h - ph))
    # fill with random noise (not zeros — zeros are unrealistic)
    patch = Image.fromarray(
        np.random.randint(0, 256, (ph, pw, 3), dtype=np.uint8)
    )
    img = img.copy()
    img.paste(patch, (x0, y0))
    return img


def augment_train(img: Image.Image) -> Image.Image:

    # ── 1. Geometric ──
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)

    if random.random() < 0.4:
        img = img.rotate(random.uniform(-15, 15),
                         resample=Image.BICUBIC, expand=False)

    # ── 2. Photometric ──
    # Wide range covers phone exposure variation
    if random.random() < 0.5:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.5, 1.5))

    if random.random() < 0.5:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.6, 1.4))

    if random.random() < 0.4:
        img = ImageEnhance.Color(img).enhance(random.uniform(0.6, 1.4))

    # Random grayscale — prevents colour-channel bias
    # LFW has B&W images so model must not rely on colour
    if random.random() < 0.05:
        img = img.convert('L').convert('RGB')

    # ── 3. Degradation ──
    # Wide blur range: covers motion blur, out-of-focus phone photos
    # Model must NOT learn 'sharp = real' — many real photos are blurry
    if random.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(
                         radius=random.uniform(0.3, 2.5)))

    # Wide JPEG range: covers WhatsApp (low quality) to original (high)
    if random.random() < 0.6:
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=random.randint(30, 95))
        buf.seek(0)
        img = Image.open(buf).convert('RGB')

    # Sharpening: covers phone portrait mode over-sharpening
    if random.random() < 0.2:
        img = ImageEnhance.Sharpness(img).enhance(random.uniform(0.0, 3.0))

    # Gaussian noise simulation
    if random.random() < 0.2:
        arr   = np.array(img, dtype=np.float32)
        noise = np.random.normal(0, random.uniform(2, 10), arr.shape)
        arr   = np.clip(arr + noise, 0, 255).astype(np.uint8)
        img   = Image.fromarray(arr)

    # ── 4. Structural ──
    # Random erasing: forces model to learn local artifacts
    # not global face statistics — key bias fix
    img = random_erase(img, p=0.3)

    return img


to_tensor_norm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Augmentation pipeline defined.')
print('Includes: flip, rotation, brightness, contrast, saturation,')
print('          grayscale, blur(0.3-2.5), JPEG(30-95), sharpening,')
print('          gaussian noise, random erasing (NEW)')

In [ ]:
class SpatialDataset(Dataset):
    def __init__(self, refs, train=True):
        self.refs  = refs
        self.train = train

    def __len__(self): return len(self.refs)

    def __getitem__(self, idx):
        r = self.refs[idx]
        try:
            img = Image.open(r.path).convert('RGB')
        except Exception:
            return (torch.zeros(3, TARGET_SIZE, TARGET_SIZE),
                    torch.tensor(0.0), r.source)

        # Step 1: resize + center crop (always)
        img  = img.resize((RESIZE_SIZE, RESIZE_SIZE), Image.BICUBIC)
        left = (RESIZE_SIZE - TARGET_SIZE) // 2
        img  = img.crop((left, left, left + TARGET_SIZE, left + TARGET_SIZE))

        # Step 2: augment at 224px (correct scale for blur/JPEG)
        if self.train:
            img = augment_train(img)

        # Step 3: tensor + ImageNet normalise
        return to_tensor_norm(img), torch.tensor(float(r.label)), r.source

print('SpatialDataset defined.')

In [ ]:
fast_val_refs = random.sample(val_refs, min(FAST_VAL_N, len(val_refs)))

train_ds    = SpatialDataset(train_refs,    train=True)
fast_val_ds = SpatialDataset(fast_val_refs, train=False)
val_ds      = SpatialDataset(val_refs,      train=False)
test_ds     = SpatialDataset(test_refs,     train=False)
celeb_ds    = SpatialDataset(celeb_refs,    train=False)

KW_T = dict(num_workers=4, pin_memory=True)
KW_E = dict(num_workers=2, pin_memory=True)

train_loader    = DataLoader(train_ds,    batch_size=BATCH_SIZE, shuffle=True,  **KW_T)
fast_val_loader = DataLoader(fast_val_ds, batch_size=BATCH_SIZE, shuffle=False, **KW_E)
val_loader      = DataLoader(val_ds,      batch_size=BATCH_SIZE, shuffle=False, **KW_E)
test_loader     = DataLoader(test_ds,     batch_size=BATCH_SIZE, shuffle=False, **KW_E)
celeb_loader    = DataLoader(celeb_ds,    batch_size=BATCH_SIZE, shuffle=False, **KW_E)

print(f'train:    {len(train_ds):,}  ({len(train_loader):,} batches/epoch)')
print(f'fast_val: {len(fast_val_ds):,}')
print(f'val:      {len(val_ds):,}')
print(f'test:     {len(test_ds):,}')
print(f'celeb:    {len(celeb_ds):,}  (held-out)')

# Sanity check
x, y, src = next(iter(train_loader))
print(f'\nBatch: {x.shape}  y: {y.unique().tolist()}')
print(f'x range: [{x.min():.2f}, {x.max():.2f}]  (expected ~[-2.1, 2.6])')
print(f'Sources in batch: {list(set(src))[:4]}')
del x, y, src

tc = Counter(r.label for r in train_refs)
print(f'\nClass balance: real={tc[0]:,}  fake={tc[1]:,}  ratio={tc[1]/max(tc[0],1):.1f}:1')
print('LFW added diversity reduces original 6:1 ratio')
print('Real diversity (not weighting) is the correct fix')

In [ ]:
def build_model():
    m = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(m.classifier[1].in_features, 1)
    )
    return m

model = build_model().to(DEVICE)
print(f'EfficientNet-B3: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')
print(f'Head: {model.classifier}')

In [ ]:
# No pos_weight — LFW provides real diversity
# Adding pos_weight on top would re-introduce fake-bias
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(1, EPOCHS - START_EPOCH), eta_min=LR_MIN
)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE=='cuda'))

print('BCEWithLogitsLoss (no pos_weight)')
print(f'AdamW lr={LR}  CosineAnnealing T_max={EPOCHS}')

In [ ]:
best_val_f1 = 0.0

if CKPT_LOAD and os.path.exists(CKPT_LOAD):
    ckpt = torch.load(CKPT_LOAD, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scheduler_state_dict' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    START_EPOCH = ckpt.get('epoch', 0)
    best_val_f1 = ckpt.get('best_val_f1', 0.0)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE=='cuda'))
    print(f'Resumed epoch {START_EPOCH}, best_val_f1={best_val_f1:.4f}')
else:
    print('Starting from scratch (CKPT_LOAD=None)')

In [ ]:
THRESHOLD = 0.50

@torch.no_grad()
def evaluate(model, loader, desc='Eval', threshold=THRESHOLD):
    model.eval()
    pa, la, sa, probs_all = [], [], [], []
    for x, y, src in tqdm(loader, desc=desc, leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE=='cuda')):
            logits = model(x).squeeze(1)
        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).long()
        pa.extend(preds.cpu().tolist())
        la.extend(y.long().cpu().tolist())
        sa.extend(src if isinstance(src,(list,tuple)) else [src])
        probs_all.extend(probs.cpu().tolist())

    ov = {
        'acc':       accuracy_score(la, pa),
        'macro_f1':  f1_score(la, pa, average='macro',  zero_division=0),
        'real_f1':   f1_score(la, pa, pos_label=0,      zero_division=0),
        'fake_f1':   f1_score(la, pa, pos_label=1,      zero_division=0),
        'real_prec': precision_score(la, pa, pos_label=0, zero_division=0),
        'real_rec':  recall_score(la, pa, pos_label=0,    zero_division=0),
        'fake_prec': precision_score(la, pa, pos_label=1, zero_division=0),
        'fake_rec':  recall_score(la, pa, pos_label=1,    zero_division=0),
        'mean_prob': float(np.mean(probs_all)),
    }
    per = {}
    for s in set(sa):
        idx = [i for i,ss in enumerate(sa) if ss==s]
        yt  = [la[i] for i in idx]
        yp  = [pa[i] for i in idx]
        pl  = 0 if yt[0]==0 else 1
        per[s] = {
            'acc':  accuracy_score(yt,yp),
            'f1':   f1_score(yt,yp,pos_label=pl,zero_division=0),
            'prec': precision_score(yt,yp,pos_label=pl,zero_division=0),
            'rec':  recall_score(yt,yp,pos_label=pl,zero_division=0),
            'n':    len(idx),
        }
    worst = min(m['f1'] for m in per.values()) if per else 0.0
    return ov, per, worst, probs_all, la


def print_eval(tag, ov, per):
    print(f'\n{tag}')
    print(f'  acc={ov["acc"]:.4f}  macro_f1={ov["macro_f1"]:.4f}  mean_prob={ov["mean_prob"]:.4f}')
    print(f'  REAL  f1={ov["real_f1"]:.4f}  prec={ov["real_prec"]:.4f}  rec={ov["real_rec"]:.4f}')
    print(f'  FAKE  f1={ov["fake_f1"]:.4f}  prec={ov["fake_prec"]:.4f}  rec={ov["fake_rec"]:.4f}')
    for s,m in sorted(per.items(), key=lambda x:x[1]['f1']):
        print(f'    {s:35s}  acc={m["acc"]:.3f}  f1={m["f1"]:.3f}  '
              f'p={m["prec"]:.3f}  r={m["rec"]:.3f}  n={m["n"]}')

print(f'evaluate() defined. Default threshold={THRESHOLD}')

In [ ]:
print(f'Training epochs {START_EPOCH+1} to {EPOCHS}')
print(f'{len(train_loader):,} batches/epoch  bs={BATCH_SIZE}')
print(f'Estimated: ~{len(train_loader)*0.8/60:.0f} min/epoch')
print('='*65)

history = []

for epoch in range(START_EPOCH, EPOCHS):
    model.train()
    run_loss = run_correct = run_n = 0

    for x, y, _ in tqdm(train_loader, desc=f'Ep {epoch+1:02d}/{EPOCHS}'):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=(DEVICE=='cuda')):
            logits = model(x).squeeze(1)
            loss   = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            preds = (torch.sigmoid(logits) >= 0.5).long()
            run_correct += (preds == y.long()).sum().item()
        run_loss += loss.item() * x.size(0)
        run_n    += x.size(0)

    scheduler.step()
    tr_loss = run_loss / max(run_n, 1)
    tr_acc  = run_correct / max(run_n, 1)
    lr_now  = scheduler.get_last_lr()[0]

    fv, _, _, _, _ = evaluate(model, fast_val_loader, desc='FastVal')

    print(f'\nEp {epoch+1:02d}/{EPOCHS}'
          f'  loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}'
          f'  macro_f1={fv["macro_f1"]:.4f}'
          f'  real_f1={fv["real_f1"]:.4f}'
          f'  fake_f1={fv["fake_f1"]:.4f}'
          f'  mean_p={fv["mean_prob"]:.3f}'
          f'  lr={lr_now:.2e}')

    ckpt = {
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'epoch':       epoch+1,
        'best_val_f1': best_val_f1,
        'train_loss':  tr_loss,
        'train_acc':   tr_acc,
        'val_macro_f1': fv['macro_f1'],
    }
    torch.save(ckpt, f'{CKPT_DIR}/epoch_{epoch+1:02d}.pth')
    print(f'  Saved: epoch_{epoch+1:02d}.pth')

    if fv['macro_f1'] > best_val_f1:
        best_val_f1 = fv['macro_f1']
        torch.save(ckpt, f'{CKPT_DIR}/best_model_spatial_v2.pth')
        print(f'  >> best_model_spatial_v2.pth  macro_f1={best_val_f1:.4f}')

    history.append({
        'epoch':epoch+1,'loss':tr_loss,'tr_acc':tr_acc,
        'macro_f1':fv['macro_f1'],'real_f1':fv['real_f1'],
        'fake_f1':fv['fake_f1'],'mean_p':fv['mean_prob'],'lr':lr_now,
    })

print('\nTraining complete.')

In [ ]:
print(f'{"Ep":>4}  {"Loss":>7}  {"TrAcc":>7}  {"MacroF1":>8}  '
      f'{"RealF1":>7}  {"FakeF1":>7}  {"MeanP":>6}  {"LR":>9}')
print('-'*72)
for h in history:
    print(f'{h["epoch"]:>4}  {h["loss"]:>7.4f}  {h["tr_acc"]:>7.4f}  '
          f'{h["macro_f1"]:>8.4f}  {h["real_f1"]:>7.4f}  '
          f'{h["fake_f1"]:>7.4f}  {h["mean_p"]:>6.3f}  {h["lr"]:>9.2e}')

In [ ]:
best_ckpt = torch.load(f'{CKPT_DIR}/best_model_spatial_v2.pth', map_location=DEVICE)
model.load_state_dict(best_ckpt['model_state_dict'])
model.to(DEVICE)
print(f'Loaded best model from epoch {best_ckpt["epoch"]}')
print(f'Saved val macro_f1: {best_ckpt["val_macro_f1"]:.4f}')

In [ ]:
print('\nFF++ Validation...')
val_ov, val_per, val_worst, _, _ = evaluate(model, val_loader, desc='Val')
print_eval('FF++ VAL', val_ov, val_per)
print(f'Worst-source F1: {val_worst:.4f}')

In [ ]:
print('\nFF++ Test...')
test_ov, test_per, test_worst, _, _ = evaluate(model, test_loader, desc='Test')
print_eval('FF++ TEST', test_ov, test_per)
print(f'Worst-source F1: {test_worst:.4f}')

In [ ]:
print('\nCelebDF (never seen during training)...')
celeb_ov, celeb_per, celeb_worst, celeb_probs, celeb_labels = evaluate(
    model, celeb_loader, desc='CelebDF'
)
print_eval('CELEBDF cross-dataset', celeb_ov, celeb_per)
fpr_c, tpr_c, _ = roc_curve(celeb_labels, celeb_probs)
celeb_auc = auc(fpr_c, tpr_c)
print(f'\nCelebDF ROC-AUC: {celeb_auc:.4f}')
print(f'Published SOTA (FF++->CelebDF): 65-76% AUC')
if celeb_auc >= 0.78:   print('>>> EXCELLENT — exceeds SOTA')
elif celeb_auc >= 0.72: print('>>> GOOD — matches SOTA')
elif celeb_auc >= 0.65: print('>>> ACCEPTABLE')
else:                   print('>>> BELOW EXPECTED — needs more training')

In [ ]:
# ══════════════════════════════════════════
#  PARETO-OPTIMAL THRESHOLD
#
#  Finds threshold satisfying BOTH targets:
#    fake_f1 >= 0.93  (catches deepfakes reliably)
#    real_f1 >= 0.88  (low false positives on real)
#
#  This IS defensible and IS standard practice:
#  - AUC (threshold-independent) is the primary metric
#  - Threshold selects operating point on the ROC curve
#  - Every published paper reports results at a chosen threshold
#  - Choosing based on Pareto criterion is MORE rigorous
#    than the arbitrary 0.5 default
# ══════════════════════════════════════════
print('Collecting val probabilities for threshold sweep...')
_, _, _, val_probs, val_labels_full = evaluate(
    model, val_loader, desc='Val probs'
)

fpr_v, tpr_v, roc_thr = roc_curve(val_labels_full, val_probs)
val_auc = auc(fpr_v, tpr_v)

FAKE_F1_TARGET = 0.93
REAL_F1_TARGET = 0.88

sweep   = np.linspace(0.05, 0.95, 500)
results = []
for thr in sweep:
    p       = [1 if pp >= thr else 0 for pp in val_probs]
    real_f1 = f1_score(val_labels_full, p, pos_label=0, zero_division=0)
    fake_f1 = f1_score(val_labels_full, p, pos_label=1, zero_division=0)
    macro   = (real_f1 + fake_f1) / 2
    penalty = (max(0, FAKE_F1_TARGET - fake_f1) +
               max(0, REAL_F1_TARGET - real_f1))
    results.append({'thr':float(thr),'real_f1':real_f1,
                    'fake_f1':fake_f1,'macro':macro,'penalty':penalty})

best = min(results, key=lambda x: (x['penalty'], -x['macro']))

j_idx      = int(np.argmax(tpr_v - fpr_v))
youden_thr = float(roc_thr[j_idx])

print('\n' + '='*60)
print(f'FF++ Val AUC:    {val_auc:.4f}')
print(f'CelebDF AUC:     {celeb_auc:.4f}')
print(f"Youden's J:      {youden_thr:.3f}")
print(f'\nPareto threshold: {best["thr"]:.3f}')
print(f'  real_f1  = {best["real_f1"]:.4f}  (target >= {REAL_F1_TARGET})')
print(f'  fake_f1  = {best["fake_f1"]:.4f}  (target >= {FAKE_F1_TARGET})')
print(f'  macro_f1 = {best["macro"]:.4f}')
print(f'  penalty  = {best["penalty"]:.4f}  (0.0 = both targets met)')
if best['penalty'] == 0:
    print('  STATUS: Both targets satisfied simultaneously — model is balanced')
else:
    print('  STATUS: Targets not fully met — consider more epochs')
print(f'\n>>> THRESHOLD = {best["thr"]:.3f} <<<')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

ax = axes[0]
ax.plot(fpr_v, tpr_v, '#3498db', lw=2, label=f'FF++ AUC={val_auc:.4f}')
ax.plot(fpr_c, tpr_c, '#e74c3c', lw=2, linestyle='--',
        label=f'CelebDF AUC={celeb_auc:.4f}')
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.4)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
thrs = [r['thr']     for r in results]
mf1s = [r['macro']   for r in results]
rf1s = [r['real_f1'] for r in results]
ff1s = [r['fake_f1'] for r in results]
ax.plot(thrs, mf1s, '#9b59b6', lw=2.5, label='Macro F1')
ax.plot(thrs, rf1s, '#27ae60', lw=1.5, linestyle='--', label='Real F1')
ax.plot(thrs, ff1s, '#e74c3c', lw=1.5, linestyle='--', label='Fake F1')
ax.axvline(best['thr'], '#9b59b6', lw=2, linestyle=':', label=f'Pareto={best["thr"]:.3f}')
ax.axhline(REAL_F1_TARGET, color='#27ae60', lw=1, linestyle=':', alpha=0.5)
ax.axhline(FAKE_F1_TARGET, color='#e74c3c', lw=1, linestyle=':', alpha=0.5)
ax.set_xlabel('Threshold'); ax.set_ylabel('F1')
ax.set_title('F1 vs Threshold', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[2]
rp = [val_probs[i] for i,l in enumerate(val_labels_full) if l==0]
fp = [val_probs[i] for i,l in enumerate(val_labels_full) if l==1]
ax.hist(rp, bins=60, alpha=0.6, color='#27ae60',
        label=f'Real (n={len(rp):,})', density=True)
ax.hist(fp, bins=60, alpha=0.6, color='#e74c3c',
        label=f'Fake (n={len(fp):,})', density=True)
ax.axvline(best['thr'], color='#9b59b6', lw=2, linestyle='--',
           label=f'Pareto={best["thr"]:.3f}')
ax.set_xlabel('Fake Probability'); ax.set_ylabel('Density')
ax.set_title('Probability Distribution\n(well-trained model: peaks near 0 and 1)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle(f'Spatial Branch v2 (From Scratch) | '
             f'FF++ AUC={val_auc:.4f}  CelebDF AUC={celeb_auc:.4f}  '
             f'Pareto thr={best["thr"]:.3f}',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/spatial_v2_analysis.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: spatial_v2_analysis.png')

In [ ]:
pareto_thr = best['thr']

print(f'Verifying Pareto threshold={pareto_thr:.3f} on FF++ TEST...')
_, _, _, test_probs, test_labels = evaluate(
    model, test_loader, desc='Test probs'
)
tp = [1 if p >= pareto_thr else 0 for p in test_probs]
print(f'  acc      = {accuracy_score(test_labels, tp):.4f}')
print(f'  macro_f1 = {f1_score(test_labels, tp, average="macro", zero_division=0):.4f}')
print(f'  real_f1  = {f1_score(test_labels, tp, pos_label=0, zero_division=0):.4f}')
print(f'  fake_f1  = {f1_score(test_labels, tp, pos_label=1, zero_division=0):.4f}')

print(f'\nVerifying on CelebDF at thr={pareto_thr:.3f}...')
cp = [1 if p >= pareto_thr else 0 for p in celeb_probs]
print(f'  acc      = {accuracy_score(celeb_labels, cp):.4f}')
print(f'  macro_f1 = {f1_score(celeb_labels, cp, average="macro", zero_division=0):.4f}')
print(f'  real_f1  = {f1_score(celeb_labels, cp, pos_label=0, zero_division=0):.4f}')
print(f'  fake_f1  = {f1_score(celeb_labels, cp, pos_label=1, zero_division=0):.4f}')
print(f'  AUC      = {celeb_auc:.4f}  (threshold-independent)')

In [ ]:
print('\n' + '='*65)
print('SPATIAL BRANCH v2 — FINAL RESULTS')
print('='*65)
print(f'Best checkpoint:   epoch {best_ckpt["epoch"]}')
print(f'Training setup:    FF++ Real + LFW (from scratch)')
print(f'Pareto threshold:  {pareto_thr:.3f}')
print(f'  Targets: fake_f1>={FAKE_F1_TARGET}  real_f1>={REAL_F1_TARGET}')
print(f'  Status:  {"BOTH MET" if best["penalty"]==0 else "BEST AVAILABLE"}')
print(f'\nFF++ Val (thr=0.50):')
print(f'  macro={val_ov["macro_f1"]:.4f}  real={val_ov["real_f1"]:.4f}  '
      f'fake={val_ov["fake_f1"]:.4f}  AUC={val_auc:.4f}')
print(f'FF++ Test (thr=0.50):')
print(f'  macro={test_ov["macro_f1"]:.4f}  real={test_ov["real_f1"]:.4f}  '
      f'fake={test_ov["fake_f1"]:.4f}')
print(f'CelebDF cross-dataset (thr=0.50):')
print(f'  macro={celeb_ov["macro_f1"]:.4f}  real={celeb_ov["real_f1"]:.4f}  '
      f'fake={celeb_ov["fake_f1"]:.4f}  AUC={celeb_auc:.4f}')
print(f'  Published SOTA: 65-76% AUC')
print(f'\nPer-manipulation F1 (FF++ Test, thr=0.50):')
for s,m in sorted(test_per.items(), key=lambda x:x[1]['f1']):
    print(f'  {s:35s}  f1={m["f1"]:.3f}  acc={m["acc"]:.3f}  n={m["n"]}')
print(f'\n>>> Save best_model_spatial_v2.pth for fusion')

In [ ]:
from IPython.display import FileLink
print(os.listdir(CKPT_DIR))
FileLink('best_model_spatial_v2.pth')